# Verifikasi Multinomial Naive Bayes Manual

**Kelompok 1 — Klasifikasi Email Ham, Phish, dan Spam**

Notebook ini menjawab dua pertanyaan:

1. **Apakah perhitungan Multinomial Naive Bayes manual sudah benar?** Model manual dibandingkan langkah demi langkah dengan `MultinomialNB` dari scikit-learn. Kedua model dilatih pada data training yang sama dengan pengaturan yang sama. Jika rumus dan kodenya benar, setiap nilai antara (jumlah kata, probabilitas, skor, prediksi, dan metrik) harus sama.
2. **Bagaimana performa model manual pada data testing yang sama dengan baseline?** Data dibersihkan dan dibagi dengan kode yang sama seperti baseline, sehingga kedua model dinilai pada email testing yang sama.

Tokenisasi, penghitungan frekuensi, pelatihan, prediksi, confusion matrix, dan metrik evaluasi pada model manual diimplementasikan tanpa library. scikit-learn hanya dipakai untuk membagi data, agar pembagiannya identik dengan baseline, dan sebagai pembanding.

| Komponen | Parameter | Nilai | Alasan |
|---|---|---|---|
| Pembagian data | `train_test_split` | 80% training / 20% testing, `stratify=y`, `random_state=42` | Kode yang sama dengan baseline, sehingga email training dan testing identik |
| Tokenisasi | pola token | `(?u)\b\w\w+\b`, huruf kecil | Sama dengan pola bawaan `CountVectorizer`, sehingga model manual dapat dibandingkan langsung dengan scikit-learn |
| Representasi | frekuensi kata | jumlah kemunculan kata (*count*), *unigram*, seluruh vocabulary | Multinomial Naive Bayes memodelkan berapa kali setiap kata muncul pada email |
| Smoothing | α | 1,0 | Laplace smoothing standar; tidak di-*tuning* |
| Toleransi pembanding | `rtol`, `atol` | 1e-9 | Penjumlahan bilangan desimal dengan urutan berbeda menghasilkan selisih pada digit terakhir. Selisih di bawah batas ini berasal dari pembulatan komputer, bukan dari rumus |

# A. Data dan Pembagian Data

## A.1 Pemuatan Dataset

Sel berikut memuat dataset dari Hugging Face dan menetapkan konfigurasi eksperimen.

In [1]:
import sys
import subprocess

try:
    from datasets import load_dataset
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets'])
    from datasets import load_dataset

import math
import re
import time
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

DATASET_NAME = 'locuoco/the-biggest-spam-ham-phish-email-dataset-300000'
LABELS = {0: 'Ham', 1: 'Phish', 2: 'Spam'}
CLASS_IDS = tuple(LABELS)
VALID_LABELS = set(LABELS)
TEST_SIZE = 0.20
SPLIT_RANDOM_STATE = 42
ALPHA = 1.0
RTOL = 1e-9
ATOL = 1e-9

dataset = load_dataset(DATASET_NAME, split='train')
raw_df = dataset.to_pandas()

print(f'Sumber dataset : {DATASET_NAME}')
print(f'Jumlah baris   : {len(raw_df):,}')
print(f'Kolom          : {raw_df.columns.tolist()}')

README.md:   0%|          | 0.00/990 [00:00<?, ?B/s]

df.csv: reconstructing file:   0%|          |  0.00B /  631MB            

df.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/365448 [00:00<?, ? examples/s]

Sumber dataset : locuoco/the-biggest-spam-ham-phish-email-dataset-300000
Jumlah baris   : 365,448
Kolom          : ['label', 'text']


## A.2 Preprocessing

Preprocessing sama dengan baseline. Langkah yang dilakukan:

1. Menghapus baris tanpa teks atau tanpa label.
2. Menghapus baris dengan label selain 0, 1, dan 2.
3. Mengganti setiap rangkaian *whitespace* (spasi ganda, tab, baris baru) dengan satu spasi, lalu menghapus teks yang kosong.
4. Menghapus seluruh baris yang teksnya sama (tanpa membedakan huruf besar dan kecil) tetapi labelnya berbeda.
5. Menyisakan satu baris untuk setiap teks yang sama dengan label yang sama.

In [2]:
data = raw_df[['text', 'label']].copy()
n_raw = len(data)

data = data.dropna(subset=['text', 'label']).copy()
n_after_missing = len(data)

data = data[data['label'].isin(VALID_LABELS)].copy()
data['label'] = data['label'].astype(int)
n_after_valid_labels = len(data)

data['clean_text'] = (
    data['text']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)
data = data[data['clean_text'].ne('')].copy()
n_after_empty_text = len(data)

data['dedup_key'] = data['clean_text'].str.lower()
labels_per_key = data.groupby('dedup_key')['label'].nunique()
conflicting_keys = labels_per_key[labels_per_key.gt(1)].index
data_without_conflicts = data.loc[~data['dedup_key'].isin(conflicting_keys)].copy()
n_after_conflicts = len(data_without_conflicts)

df_clean = (
    data_without_conflicts
    .drop_duplicates(subset=['dedup_key'], keep='first')
    [['clean_text', 'label']]
    .rename(columns={'clean_text': 'text'})
    .reset_index(drop=True)
)
n_final = len(df_clean)
del data, data_without_conflicts

display(pd.DataFrame({
    'tahap': [
        'Data awal', 'Setelah baris kosong dihapus', 'Setelah validasi label',
        'Setelah teks kosong dihapus', 'Setelah teks dengan label berbeda dihapus',
        'Setelah duplikat dihapus'
    ],
    'jumlah_data_tersisa': [
        n_raw, n_after_missing, n_after_valid_labels,
        n_after_empty_text, n_after_conflicts, n_final
    ]
}))

,tahap,jumlah_data_tersisa
0,Data awal,365448
1,Setelah baris kosong dihapus,365446
2,Setelah validasi label,365446
3,Setelah teks kosong dihapus,365445
4,Setelah teks dengan label berbeda dihapus,365401
5,Setelah duplikat dihapus,278941


## A.3 Pembagian Data Training dan Testing

Data dibagi dengan `train_test_split` yang sama seperti baseline. Data bersih, urutan baris, proporsi testing, dan `random_state` sama, sehingga email yang masuk ke data training dan data testing juga sama. Sidik jari (*hash*) indeks data testing ditampilkan agar pembagian ini dapat dicocokkan dengan notebook lain.

In [3]:
from sklearn.model_selection import train_test_split

X = df_clean['text']
y = df_clean['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=SPLIT_RANDOM_STATE,
    stratify=y
)

train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df = pd.DataFrame({'text': X_test, 'label': y_test})

test_fingerprint = hashlib.sha256(
    ','.join(map(str, sorted(X_test.index))).encode()
).hexdigest()[:16]

split_table = pd.DataFrame({
    'Training (jumlah)': y_train.value_counts().sort_index(),
    'Training (%)': (y_train.value_counts(normalize=True).sort_index() * 100).round(4),
    'Testing (jumlah)': y_test.value_counts().sort_index(),
    'Testing (%)': (y_test.value_counts(normalize=True).sort_index() * 100).round(4),
}).rename(index=LABELS).rename_axis('kelas')
display(split_table)
print(f'Data training        : {len(X_train):,} email')
print(f'Data testing         : {len(X_test):,} email')
print(f'Sidik jari data testing: {test_fingerprint}')

,Training (jumlah),Training (%),Testing (jumlah),Testing (%)
kelas,,,,
Ham,102192,45.7948,25548,45.7940
Phish,34270,15.3572,8568,15.3579
Spam,86690,38.8480,21673,38.8482


Data training        : 223,152 email
Data testing         : 55,789 email
Sidik jari data testing: 193b7f8793006d67


# B. Model Multinomial Naive Bayes Manual

## B.1 Tokenisasi dan Perhitungan Probabilitas

**Tokenisasi** memecah teks email menjadi unit kecil (token) agar dapat dihitung. Seluruh huruf diubah menjadi huruf kecil, tanda baca hanya menjadi pemisah, dan token minimal terdiri dari dua karakter huruf, angka, atau *underscore*. Pola yang dipakai adalah `(?u)\b\w\w+\b`. **Vocabulary** adalah daftar seluruh token unik pada data training. Token pada data testing yang tidak ada dalam vocabulary diabaikan.

Untuk suatu kelas $c$, total dokumen training $D$, dan total dokumen pada kelas tersebut $D_c$, **probabilitas prior** adalah:

$$
P(c)=\frac{D_c}{D}
$$

Dengan $F_{w,c}$ adalah frekuensi token $w$ pada seluruh dokumen training kelas $c$, $T_c$ adalah total token pada kelas $c$, dan $\vert{}V\vert{}$ adalah ukuran vocabulary training, **probabilitas likelihood** dengan Laplace smoothing adalah:

$$
P(w\mid c)=\frac{F_{w,c}+\alpha}{T_c+\alpha|V|}
$$

Berdasarkan frekuensi token $f(w,x)$ dalam email $x$, skor kelas dihitung dalam bentuk logaritma agar tidak terjadi *underflow*:

$$
\log S_c=\log P(c)+\sum_{w\in V}f(w,x)\log P(w\mid c)
$$

Kelas dengan $\log S_c$ tertinggi menjadi prediksi. Sel berikut mendefinisikan fungsi tokenisasi dan pelatihan, lalu melatih model pada data training.

In [4]:
TOKEN_PATTERN = re.compile(r'(?u)\b\w\w+\b')

def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())

def word_frequencies(tokens):
    frequencies = {}
    for token in tokens:
        frequencies[token] = frequencies.get(token, 0) + 1
    return frequencies

def train_multinomial_nb(frame, alpha=1.0):
    document_count = len(frame)
    class_document_count = {class_id: 0 for class_id in CLASS_IDS}
    class_token_count = {class_id: 0 for class_id in CLASS_IDS}
    token_count = {class_id: {} for class_id in CLASS_IDS}
    vocabulary = set()
    for text, class_id in frame[['text', 'label']].itertuples(index=False, name=None):
        class_document_count[class_id] += 1
        for token in tokenize(text):
            vocabulary.add(token)
            class_token_count[class_id] += 1
            counts = token_count[class_id]
            counts[token] = counts.get(token, 0) + 1
    vocabulary_size = len(vocabulary)
    if document_count == 0 or vocabulary_size == 0:
        raise ValueError('Training set atau vocabulary kosong.')
    if any(class_document_count[class_id] == 0 for class_id in CLASS_IDS):
        raise ValueError('Semua kelas wajib tersedia pada training set.')
    log_prior = {class_id: math.log(class_document_count[class_id] / document_count) for class_id in CLASS_IDS}
    likelihood_denominator = {
        class_id: class_token_count[class_id] + alpha * vocabulary_size
        for class_id in CLASS_IDS
    }
    return {
        'alpha': alpha, 'D': document_count,
        'class_document_count': class_document_count,
        'class_token_count': class_token_count, 'token_count': token_count,
        'vocabulary': vocabulary, 'V': vocabulary_size,
        'log_prior': log_prior, 'likelihood_denominator': likelihood_denominator,
    }

start = time.time()
model = train_multinomial_nb(train_df, ALPHA)
manual_train_seconds = time.time() - start

training_summary = pd.DataFrame([
    {
        'kelas': LABELS[class_id],
        'D_c': model['class_document_count'][class_id],
        'P(c)': model['class_document_count'][class_id] / model['D'],
        'T_c': model['class_token_count'][class_id],
        'T_c + alpha|V|': model['likelihood_denominator'][class_id],
    }
    for class_id in CLASS_IDS
])
print(f"Dokumen training : {model['D']:,}")
print(f"Ukuran vocabulary: {model['V']:,}")
print(f'Waktu pelatihan  : {manual_train_seconds:.1f} detik')
display(training_summary)

Dokumen training : 223,152
Ukuran vocabulary: 721,764
Waktu pelatihan  : 64.5 detik


,kelas,D_c,P(c),T_c,T_c + alpha|V|
0,Ham,102192,0.457948,33128748,33850512.0
1,Phish,34270,0.153572,3991326,4713090.0
2,Spam,86690,0.388480,17003718,17725482.0


## B.2 Verifikasi dengan Hitungan Tangan

Sebelum dibandingkan pada data asli, rumus diperiksa pada enam dokumen kecil yang dapat dihitung dengan tangan. Terdapat dua dokumen untuk setiap kelas, sembilan kata unik, dan empat token pada setiap kelas. Untuk teks uji `verify password`:

$$
P(\mathrm{Phish})=\frac{2}{6},\quad P(\mathrm{verify}\mid\mathrm{Phish})=\frac{2+1}{4+9}=\frac{3}{13},\quad P(\mathrm{password}\mid\mathrm{Phish})=\frac{1+1}{4+9}=\frac{2}{13}
$$

$$
S_{\mathrm{Phish}}=\frac{2}{6}\times\frac{3}{13}\times\frac{2}{13}=\frac{12}{1014},\qquad
S_{\mathrm{Ham}}=S_{\mathrm{Spam}}=\frac{2}{6}\times\frac{1}{13}\times\frac{1}{13}=\frac{2}{1014}
$$

Setelah dinormalisasi, $P(\mathrm{Phish}\mid x)=12/(12+2+2)=0{,}75$ dan $P(\mathrm{Ham}\mid x)=P(\mathrm{Spam}\mid x)=0{,}125$. Sel berikut mendefinisikan fungsi prediksi, lalu membandingkan hasil kode dengan hitungan tangan tersebut.

In [5]:
def predict_multinomial_nb(text, fitted, include_probabilities=False):
    frequencies = word_frequencies(tokenize(text))
    known_frequencies = {token: frequency for token, frequency in frequencies.items() if token in fitted['vocabulary']}
    log_scores = {}
    for class_id in CLASS_IDS:
        score = fitted['log_prior'][class_id]
        denominator = fitted['likelihood_denominator'][class_id]
        counts = fitted['token_count'][class_id]
        for token, frequency in known_frequencies.items():
            numerator = counts.get(token, 0) + fitted['alpha']
            score += frequency * math.log(numerator / denominator)
        log_scores[class_id] = score
    predicted_class = max(CLASS_IDS, key=lambda class_id: log_scores[class_id])
    probabilities = None
    if include_probabilities:
        maximum = max(log_scores.values())
        shifted = {class_id: math.exp(log_scores[class_id] - maximum) for class_id in CLASS_IDS}
        normalizer = sum(shifted.values())
        probabilities = {class_id: shifted[class_id] / normalizer for class_id in CLASS_IDS}
    return predicted_class, log_scores, probabilities, known_frequencies

toy_df = pd.DataFrame([
    {'text': 'meeting schedule', 'label': 0},
    {'text': 'meeting plan', 'label': 0},
    {'text': 'verify account', 'label': 1},
    {'text': 'verify password', 'label': 1},
    {'text': 'win money', 'label': 2},
    {'text': 'win prize', 'label': 2},
])
toy_model = train_multinomial_nb(toy_df, ALPHA)
toy_pred, toy_scores, toy_probabilities, _ = predict_multinomial_nb(
    'verify password', toy_model, include_probabilities=True
)

hand_scores = {0: 2 / 1014, 1: 12 / 1014, 2: 2 / 1014}
hand_posteriors = {0: 0.125, 1: 0.750, 2: 0.125}
toy_table = pd.DataFrame([
    {
        'kelas': LABELS[class_id],
        'log S_c (tangan)': math.log(hand_scores[class_id]),
        'log S_c (kode)': toy_scores[class_id],
        'posterior (tangan)': hand_posteriors[class_id],
        'posterior (kode)': toy_probabilities[class_id],
        'sesuai': 'Ya' if (
            math.isclose(toy_scores[class_id], math.log(hand_scores[class_id]), rel_tol=RTOL, abs_tol=ATOL)
            and math.isclose(toy_probabilities[class_id], hand_posteriors[class_id], rel_tol=RTOL, abs_tol=ATOL)
        ) else 'Tidak',
    }
    for class_id in CLASS_IDS
])
display(toy_table)

toy_ok = toy_model['V'] == 9 and toy_pred == 1 and toy_table['sesuai'].eq('Ya').all()
display(Markdown(
    f"Ukuran vocabulary dari kode: {toy_model['V']} kata; prediksi kode: **{LABELS[toy_pred]}**. "
    + ('Seluruh nilai sama dengan hitungan tangan.' if toy_ok
       else '**Terdapat nilai yang berbeda dari hitungan tangan.** Periksa tabel di atas.')
))

,kelas,log S_c (tangan),log S_c (kode),posterior (tangan),posterior (kode),sesuai
0,Ham,-6.228511,-6.228511,0.125,0.125,Ya
1,Phish,-4.436752,-4.436752,0.750,0.750,Ya
2,Spam,-6.228511,-6.228511,0.125,0.125,Ya


Ukuran vocabulary dari kode: 9 kata; prediksi kode: **Phish**. Seluruh nilai sama dengan hitungan tangan.

## B.3 Prediksi pada Data Testing

Sel berikut menghitung $\log S_c$ dan probabilitas posterior ketiga kelas untuk setiap email testing. Posterior dihitung dengan normalisasi *log-sum-exp* agar tetap stabil walaupun skornya sangat negatif. Seluruh skor disimpan untuk dibandingkan dengan scikit-learn pada bagian C.

In [6]:
start = time.time()
manual_pred = []
manual_log_scores = []
manual_posteriors = []
for text in test_df['text']:
    predicted, log_scores, probabilities, _ = predict_multinomial_nb(text, model, include_probabilities=True)
    manual_pred.append(predicted)
    manual_log_scores.append([log_scores[class_id] for class_id in CLASS_IDS])
    manual_posteriors.append([probabilities[class_id] for class_id in CLASS_IDS])
manual_predict_seconds = time.time() - start

manual_pred = np.array(manual_pred)
manual_log_scores = np.array(manual_log_scores)
manual_posteriors = np.array(manual_posteriors)

results_df = test_df.copy()
results_df['predicted_label'] = manual_pred
print(f'Prediksi selesai: {len(results_df):,} email testing ({manual_predict_seconds:.1f} detik)')

Prediksi selesai: 55,789 email testing (24.3 detik)


# C. Verifikasi terhadap scikit-learn

Model pembanding adalah `CountVectorizer` dengan pola token dan huruf kecil yang sama, diikuti `MultinomialNB(alpha=1.0)`. scikit-learn menghitung rumus yang sama dengan cara yang berbeda (matriks jarang dan operasi vektor), sehingga kesamaan hasil menunjukkan bahwa kode manual benar.

Pemeriksaan disusun mengikuti urutan perhitungan. Jika suatu pemeriksaan gagal, letak kesalahannya dapat ditemukan dari langkah pertama yang tidak sesuai:

1. **Vocabulary**: ukuran dan isi daftar kata sama (tokenisasi benar).
2. **$D_c$**: jumlah email training per kelas sama.
3. **$T_c$**: total token per kelas sama.
4. **$F_{w,c}$**: frekuensi setiap kata pada setiap kelas sama (penghitungan benar).
5. **$\log P(c)$**: log prior sama.
6. **$\log P(w\mid c)$**: log likelihood setiap kata sama (rumus smoothing benar).
7. **$\log S_c$**: skor setiap email testing sama (rumus prediksi benar).
8. **Posterior**: $P(c\mid x)$ setiap email testing sama (normalisasi benar).
9. **Prediksi**: kelas hasil prediksi setiap email testing sama.

Nilai bilangan bulat harus sama persis. Nilai desimal dianggap sama jika selisihnya di bawah toleransi `rtol` dan `atol`.

Sel berikut melatih model pembanding pada data training yang sama.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

start = time.time()
count_vectorizer = CountVectorizer(lowercase=True, token_pattern=r'(?u)\b\w\w+\b')
X_train_counts = count_vectorizer.fit_transform(X_train)
sk_model = MultinomialNB(alpha=ALPHA)
sk_model.fit(X_train_counts, y_train)
sk_train_seconds = time.time() - start

start = time.time()
X_test_counts = count_vectorizer.transform(X_test)
sk_log_scores = sk_model.predict_joint_log_proba(X_test_counts)
sk_posteriors = sk_model.predict_proba(X_test_counts)
sk_pred = sk_model.classes_[sk_log_scores.argmax(axis=1)]
sk_predict_seconds = time.time() - start

print(f'Kelas scikit-learn : {sk_model.classes_.tolist()}')
print(f'Ukuran vocabulary  : {len(count_vectorizer.vocabulary_):,}')
print(f'Waktu pelatihan    : {sk_train_seconds:.1f} detik')
print(f'Waktu prediksi     : {sk_predict_seconds:.1f} detik')

Sel berikut menjalankan sembilan pemeriksaan dan menampilkan hasilnya dalam satu tabel.

In [8]:
feature_names = count_vectorizer.get_feature_names_out()
sk_vocabulary = set(feature_names)
only_manual = len(model['vocabulary'] - sk_vocabulary)
only_sklearn = len(sk_vocabulary - model['vocabulary'])

manual_feature_count = np.array([
    [model['token_count'][class_id].get(word, 0) for word in feature_names]
    for class_id in CLASS_IDS
], dtype=np.float64)
manual_denominator = np.array([model['likelihood_denominator'][class_id] for class_id in CLASS_IDS])
manual_log_likelihood = np.log((manual_feature_count + ALPHA) / manual_denominator[:, None])
manual_log_prior = np.array([model['log_prior'][class_id] for class_id in CLASS_IDS])
manual_doc_count = np.array([model['class_document_count'][class_id] for class_id in CLASS_IDS])
manual_token_total = np.array([model['class_token_count'][class_id] for class_id in CLASS_IDS])

def float_check(name, manual_values, sklearn_values):
    manual_values = np.asarray(manual_values, dtype=np.float64)
    sklearn_values = np.asarray(sklearn_values, dtype=np.float64)
    difference = np.abs(manual_values - sklearn_values)
    relative = difference / np.maximum(np.abs(sklearn_values), 1.0)
    return {
        'pemeriksaan': name,
        'jumlah nilai': manual_values.size,
        'nilai berbeda': int((~np.isclose(manual_values, sklearn_values, rtol=RTOL, atol=ATOL)).sum()),
        'selisih absolut maks.': float(difference.max()),
        'selisih relatif maks.': float(relative.max()),
    }

def exact_check(name, manual_values, sklearn_values):
    manual_values = np.asarray(manual_values)
    sklearn_values = np.asarray(sklearn_values)
    difference = np.abs(manual_values.astype(np.float64) - sklearn_values.astype(np.float64))
    return {
        'pemeriksaan': name,
        'jumlah nilai': manual_values.size,
        'nilai berbeda': int((manual_values != sklearn_values).sum()),
        'selisih absolut maks.': float(difference.max()),
        'selisih relatif maks.': np.nan,
    }

checks = [
    {
        'pemeriksaan': '1. Vocabulary (|V| dan isi)',
        'jumlah nilai': model['V'],
        'nilai berbeda': only_manual + only_sklearn,
        'selisih absolut maks.': float(abs(model['V'] - len(sk_vocabulary))),
        'selisih relatif maks.': np.nan,
    },
    exact_check('2. D_c (email per kelas)', manual_doc_count, sk_model.class_count_),
    exact_check('3. T_c (token per kelas)', manual_token_total, sk_model.feature_count_.sum(axis=1)),
    exact_check('4. F_w,c (frekuensi kata per kelas)', manual_feature_count, sk_model.feature_count_),
    float_check('5. log P(c)', manual_log_prior, sk_model.class_log_prior_),
    float_check('6. log P(w|c)', manual_log_likelihood, sk_model.feature_log_prob_),
    float_check('7. log S_c (email testing)', manual_log_scores, sk_log_scores),
    float_check('8. Posterior P(c|x) (email testing)', manual_posteriors, sk_posteriors),
    exact_check('9. Prediksi kelas (email testing)', manual_pred, sk_pred),
]
check_table = pd.DataFrame(checks)
check_table['sesuai'] = np.where(check_table['nilai berbeda'].eq(0), 'Ya', 'Tidak')
pd.set_option('display.float_format', lambda value: f'{value:.3g}')
display(check_table)
pd.reset_option('display.float_format')
print(f'Kata hanya di model manual : {only_manual:,}')
print(f'Kata hanya di scikit-learn : {only_sklearn:,}')

,pemeriksaan,jumlah nilai,nilai berbeda,selisih absolut maks.,selisih relatif maks.,sesuai
0,1. Vocabulary (|V| dan isi),721764,0,0,NaN,Ya
1,2. D_c (email per kelas),3,0,0,NaN,Ya
2,3. T_c (token per kelas),3,0,0,NaN,Ya
3,"4. F_w,c (frekuensi kata per kelas)",2165292,0,0,NaN,Ya
4,5. log P(c),3,0,7.77e-16,7.77e-16,Ya
5,6. log P(w|c),2165292,0,3.55e-15,5.27e-16,Ya
6,7. log S_c (email testing),167367,0,8.29e-08,3.29e-14,Ya
7,8. Posterior P(c|x) (email testing),167367,0,2.53e-12,2.53e-12,Ya
8,9. Prediksi kelas (email testing),55789,0,0,NaN,Ya


Kata hanya di model manual : 0
Kata hanya di scikit-learn : 0


Sel berikut menuliskan kesimpulan verifikasi. Jika ada prediksi yang berbeda, email tersebut ditampilkan bersama selisih skor antara dua kelas teratas, untuk melihat apakah perbedaan hanya terjadi pada skor yang hampir seri.

In [9]:
failed_checks = check_table.loc[check_table['sesuai'].eq('Tidak'), 'pemeriksaan'].tolist()
max_score_difference = check_table.loc[check_table['pemeriksaan'].str.startswith('7.'), 'selisih absolut maks.'].iloc[0]
max_relative_difference = check_table.loc[check_table['pemeriksaan'].str.startswith('7.'), 'selisih relatif maks.'].iloc[0]
n_prediction_differences = int((manual_pred != sk_pred).sum())

if not failed_checks:
    display(Markdown(
        f'**Kesimpulan verifikasi: perhitungan manual benar.** Kesembilan pemeriksaan sesuai. '
        f'Vocabulary, jumlah email, dan frekuensi kata sama persis dengan scikit-learn. '
        f'Selisih terbesar pada skor $\\log S_c$ adalah {max_score_difference:.2e} '
        f'(relatif {max_relative_difference:.2e}), yaitu selisih pembulatan komputer. '
        f'Seluruh {len(manual_pred):,} prediksi pada data testing sama.'
    ))
else:
    display(Markdown(
        f'**Kesimpulan verifikasi: terdapat perbedaan.** Pemeriksaan yang tidak sesuai: '
        f'{", ".join(failed_checks)}. Kesalahan berada pada langkah pertama yang tidak sesuai, '
        f'karena langkah berikutnya bergantung padanya. Prediksi yang berbeda: {n_prediction_differences:,} email.'
    ))

if n_prediction_differences:
    different = np.flatnonzero(manual_pred != sk_pred)
    sorted_scores = np.sort(sk_log_scores[different], axis=1)
    display(pd.DataFrame({
        'aktual': test_df['label'].iloc[different].map(LABELS).values,
        'prediksi manual': pd.Series(manual_pred[different]).map(LABELS).values,
        'prediksi scikit-learn': pd.Series(sk_pred[different]).map(LABELS).values,
        'selisih dua skor teratas': sorted_scores[:, -1] - sorted_scores[:, -2],
        'cuplikan teks': test_df['text'].iloc[different].str[:120].values,
    }).head(20))

**Kesimpulan verifikasi: perhitungan manual benar.** Kesembilan pemeriksaan sesuai. Vocabulary, jumlah email, dan frekuensi kata sama persis dengan scikit-learn. Selisih terbesar pada skor $\log S_c$ adalah 8.29e-08 (relatif 3.29e-14), yaitu selisih pembulatan komputer. Seluruh 55,789 prediksi pada data testing sama.

# D. Evaluasi pada Data Testing

## D.1 Confusion Matrix dan Metrik Manual

Untuk setiap kelas, terdapat empat komponen confusion matrix:

- $TP$ (*True Positive*): email pada kelas tersebut diprediksi dengan benar.
- $FP$ (*False Positive*): email dari kelas lain diprediksi masuk ke kelas tersebut.
- $FN$ (*False Negative*): email pada kelas tersebut diprediksi masuk ke kelas lain.
- $TN$ (*True Negative*): email dari kelas lain tidak diprediksi masuk ke kelas tersebut.

$$
Precision_c=\frac{TP_c}{TP_c+FP_c},\quad Recall_c=\frac{TP_c}{TP_c+FN_c},\quad
Specificity_c=\frac{TN_c}{TN_c+FP_c},\quad F1_c=\frac{2\,Precision_c\,Recall_c}{Precision_c+Recall_c}
$$

*Macro average* memberi bobot sama pada setiap kelas, *weighted average* memberi bobot sesuai jumlah email per kelas (*support*), dan *micro average* menghitung dari total TP, FP, dan FN. *Balanced accuracy* adalah rata-rata recall ketiga kelas. *Cohen's kappa* membandingkan akurasi dengan akurasi yang diharapkan secara kebetulan, dan *Matthews Correlation Coefficient* (MCC) mengukur korelasi antara label aktual dan prediksi.

Sel berikut menghitung seluruh metrik tanpa library.

In [10]:
confusion = {actual: {predicted: 0 for predicted in CLASS_IDS} for actual in CLASS_IDS}
for actual, predicted in results_df[['label', 'predicted_label']].itertuples(index=False, name=None):
    confusion[actual][predicted] += 1
confusion_df = pd.DataFrame(
    [[confusion[actual][predicted] for predicted in CLASS_IDS] for actual in CLASS_IDS],
    index=['Aktual ' + LABELS[class_id] for class_id in CLASS_IDS],
    columns=['Prediksi ' + LABELS[class_id] for class_id in CLASS_IDS],
)
display(confusion_df.rename_axis('Aktual / Prediksi').reset_index())

def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0

n_test = len(results_df)
per_class_rows = []
for class_id in CLASS_IDS:
    tp = confusion[class_id][class_id]
    fn = sum(confusion[class_id][other] for other in CLASS_IDS if other != class_id)
    fp = sum(confusion[other][class_id] for other in CLASS_IDS if other != class_id)
    tn = n_test - tp - fn - fp
    precision = safe_divide(tp, tp + fp)
    recall = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    f1 = safe_divide(2 * precision * recall, precision + recall)
    per_class_rows.append({
        'Class': LABELS[class_id], 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Precision': precision, 'Recall': recall, 'Specificity': specificity,
        'F1-score': f1, 'Support': tp + fn,
    })
per_class_metrics = pd.DataFrame(per_class_rows).set_index('Class')
display(per_class_metrics.reset_index().round(4))

correct = sum(confusion[class_id][class_id] for class_id in CLASS_IDS)
accuracy = safe_divide(correct, n_test)
macro_precision = per_class_metrics['Precision'].mean()
macro_recall = per_class_metrics['Recall'].mean()
macro_specificity = per_class_metrics['Specificity'].mean()
macro_f1 = per_class_metrics['F1-score'].mean()
weighted_precision = (per_class_metrics['Precision'] * per_class_metrics['Support']).sum() / n_test
weighted_recall = (per_class_metrics['Recall'] * per_class_metrics['Support']).sum() / n_test
weighted_f1 = (per_class_metrics['F1-score'] * per_class_metrics['Support']).sum() / n_test
micro_tp = int(per_class_metrics['TP'].sum())
micro_fp = int(per_class_metrics['FP'].sum())
micro_fn = int(per_class_metrics['FN'].sum())
micro_precision = safe_divide(micro_tp, micro_tp + micro_fp)
micro_recall = safe_divide(micro_tp, micro_tp + micro_fn)
micro_f1 = safe_divide(2 * micro_precision * micro_recall, micro_precision + micro_recall)
row_totals = [sum(confusion[c].values()) for c in CLASS_IDS]
column_totals = [sum(confusion[a][c] for a in CLASS_IDS) for c in CLASS_IDS]
expected_agreement = sum(row_totals[i] * column_totals[i] for i in range(len(CLASS_IDS))) / (n_test ** 2)
cohen_kappa = safe_divide(accuracy - expected_agreement, 1 - expected_agreement)
mcc_numerator = correct * n_test - sum(row_totals[i] * column_totals[i] for i in range(len(CLASS_IDS)))
mcc_denominator = math.sqrt(
    (n_test ** 2 - sum(total ** 2 for total in column_totals))
    * (n_test ** 2 - sum(total ** 2 for total in row_totals))
)
mcc = safe_divide(mcc_numerator, mcc_denominator)
manual_metrics = {
    'Accuracy': accuracy, 'Balanced Accuracy': macro_recall,
    'Macro Precision': macro_precision, 'Macro Recall': macro_recall,
    'Macro Specificity': macro_specificity, 'Macro F1': macro_f1,
    'Weighted Precision': weighted_precision, 'Weighted Recall': weighted_recall,
    'Weighted F1': weighted_f1, 'Micro Precision': micro_precision,
    'Micro Recall': micro_recall, 'Micro F1': micro_f1,
    "Cohen's Kappa": cohen_kappa, 'Multiclass MCC': mcc,
}
display(pd.DataFrame(list(manual_metrics.items()), columns=['Metric', 'Result']).round(4))

,Aktual / Prediksi,Prediksi Ham,Prediksi Phish,Prediksi Spam
0,Aktual Ham,25118,76,354
1,Aktual Phish,24,8198,346
2,Aktual Spam,1301,1157,19215


,Class,TP,FP,FN,TN,Precision,Recall,Specificity,F1-score,Support
0,Ham,25118,1325,430,28916,0.9499,0.9832,0.9562,0.9662,25548
1,Phish,8198,1233,370,45988,0.8693,0.9568,0.9739,0.9109,8568
2,Spam,19215,700,2458,33416,0.9649,0.8866,0.9795,0.9241,21673


,Metric,Result
0,Accuracy,0.9416
1,Balanced Accuracy,0.9422
2,Macro Precision,0.9280
3,Macro Recall,0.9422
4,Macro Specificity,0.9699
5,Macro F1,0.9337
6,Weighted Precision,0.9433
7,Weighted Recall,0.9416
8,Weighted F1,0.9414
9,Micro Precision,0.9416


## D.2 Verifikasi Metrik terhadap scikit-learn

Sel berikut menghitung metrik yang sama dengan fungsi scikit-learn dari prediksi manual yang sama. Jika kode metrik manual benar, kedua kolom bernilai sama. *Macro specificity* tidak tersedia di scikit-learn, sehingga tidak ikut dibandingkan.

In [11]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, cohen_kappa_score, confusion_matrix,
    f1_score, matthews_corrcoef, precision_score, recall_score
)

y_true = results_df['label'].to_numpy()
sk_metrics = {
    'Accuracy': accuracy_score(y_true, manual_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_true, manual_pred),
    'Macro Precision': precision_score(y_true, manual_pred, average='macro', zero_division=0),
    'Macro Recall': recall_score(y_true, manual_pred, average='macro', zero_division=0),
    'Macro F1': f1_score(y_true, manual_pred, average='macro', zero_division=0),
    'Weighted Precision': precision_score(y_true, manual_pred, average='weighted', zero_division=0),
    'Weighted Recall': recall_score(y_true, manual_pred, average='weighted', zero_division=0),
    'Weighted F1': f1_score(y_true, manual_pred, average='weighted', zero_division=0),
    'Micro Precision': precision_score(y_true, manual_pred, average='micro', zero_division=0),
    'Micro Recall': recall_score(y_true, manual_pred, average='micro', zero_division=0),
    'Micro F1': f1_score(y_true, manual_pred, average='micro', zero_division=0),
    "Cohen's Kappa": cohen_kappa_score(y_true, manual_pred),
    'Multiclass MCC': matthews_corrcoef(y_true, manual_pred),
}
metric_check = pd.DataFrame({
    'Metric': list(sk_metrics),
    'Manual': [manual_metrics[name] for name in sk_metrics],
    'scikit-learn': list(sk_metrics.values()),
})
metric_check['Selisih'] = (metric_check['Manual'] - metric_check['scikit-learn']).abs()
metric_check['Sesuai'] = np.where(
    np.isclose(metric_check['Manual'], metric_check['scikit-learn'], rtol=RTOL, atol=ATOL), 'Ya', 'Tidak'
)
display(metric_check)

sk_confusion = confusion_matrix(y_true, manual_pred, labels=list(CLASS_IDS))
confusion_differences = int((confusion_df.to_numpy() != sk_confusion).sum())
metrics_ok = metric_check['Sesuai'].eq('Ya').all() and confusion_differences == 0
display(Markdown(
    f'Sel confusion matrix yang berbeda dari scikit-learn: {confusion_differences}. '
    + ('Seluruh metrik manual sama dengan scikit-learn.' if metrics_ok
       else '**Terdapat metrik yang berbeda dari scikit-learn.** Periksa tabel di atas.')
))

,Metric,Manual,scikit-learn,Selisih,Sesuai
0,Accuracy,0.941601,0.941601,0.000000e+00,Ya
1,Balanced Accuracy,0.942191,0.942191,0.000000e+00,Ya
2,Macro Precision,0.928001,0.928001,0.000000e+00,Ya
3,Macro Recall,0.942191,0.942191,0.000000e+00,Ya
4,Macro F1,0.933749,0.933749,1.110223e-16,Ya
5,Weighted Precision,0.943320,0.943320,0.000000e+00,Ya
6,Weighted Recall,0.941601,0.941601,0.000000e+00,Ya
7,Weighted F1,0.941365,0.941365,1.110223e-16,Ya
8,Micro Precision,0.941601,0.941601,0.000000e+00,Ya
9,Micro Recall,0.941601,0.941601,0.000000e+00,Ya


Sel confusion matrix yang berbeda dari scikit-learn: 0. Seluruh metrik manual sama dengan scikit-learn.

## D.3 Perbandingan dengan Baseline pada Data Testing yang Sama

Baseline (TF-IDF *unigram* dengan `sublinear_tf=True`, seluruh vocabulary, dan `MultinomialNB(alpha=1.0)`) dilatih ulang pada data training yang sama. Hasilnya dibandingkan dengan nilai yang tercatat pada notebook baseline. Jika sama, pembagian data pada notebook ini terbukti identik dengan baseline.

Model manual tidak dipakai untuk memilih model atau parameter. Nilai α = 1 ditetapkan sebelum hasil terlihat, sehingga penilaian pada data testing tidak memengaruhi keputusan apa pun. Kedua model membaca teks hasil preprocessing di atas, tanpa normalisasi angka dan *stopword*, sehingga keduanya masih dapat memanfaatkan perbedaan format antarsumber data. Oleh karena itu, skor di bawah ini tidak dibandingkan dengan model akhir yang membaca teks V3.

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

BASELINE_RECORDED = {'Accuracy': 0.8881, 'Macro F1': 0.8377, 'Phish Recall': 0.5316}

baseline_model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, ngram_range=(1, 1), sublinear_tf=True)),
    ('classifier', MultinomialNB(alpha=1.0)),
])
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

def summarize(predictions):
    matrix = confusion_matrix(y_true, predictions, labels=list(CLASS_IDS))
    recalls = recall_score(y_true, predictions, labels=list(CLASS_IDS), average=None)
    return {
        'Accuracy': accuracy_score(y_true, predictions),
        'Macro F1': f1_score(y_true, predictions, average='macro'),
        'Ham Recall': recalls[0],
        'Phish Recall': recalls[1],
        'Spam Recall': recalls[2],
        'Ham hilang (Ham → Phish/Spam)': int(matrix[0, 1] + matrix[0, 2]),
        'Phish → Ham': int(matrix[1, 0]),
        'Spam → Ham': int(matrix[2, 0]),
        'Phish → Spam': int(matrix[1, 2]),
    }

baseline_summary = summarize(baseline_pred)
manual_summary = summarize(manual_pred)

reproduction = pd.DataFrame({
    'Metric': list(BASELINE_RECORDED),
    'Tercatat pada baseline': list(BASELINE_RECORDED.values()),
    'Dilatih ulang di sini': [round(baseline_summary[name], 4) for name in BASELINE_RECORDED],
})
reproduction['Sama'] = np.where(
    np.isclose(reproduction['Tercatat pada baseline'], reproduction['Dilatih ulang di sini'], atol=5e-5), 'Ya', 'Tidak'
)
display(reproduction)
split_identical = reproduction['Sama'].eq('Ya').all()
display(Markdown(
    'Baseline yang dilatih ulang menghasilkan nilai yang sama dengan catatan baseline, '
    'sehingga data training dan data testing pada notebook ini identik dengan baseline.'
    if split_identical else
    '**Baseline yang dilatih ulang tidak sama dengan catatan baseline.** Pembagian data atau versi library mungkin berbeda; '
    'perbandingan di bawah tetap adil karena kedua model memakai data yang sama di notebook ini.'
))

comparison = pd.DataFrame({
    'Baseline (TF-IDF + MNB)': baseline_summary,
    'MNB manual (count)': manual_summary,
})
comparison['Selisih (manual − baseline)'] = comparison['MNB manual (count)'] - comparison['Baseline (TF-IDF + MNB)']
display(comparison.rename_axis('Metric').reset_index().round(4))

,Metric,Tercatat pada baseline,Dilatih ulang di sini,Sama
0,Accuracy,0.8881,0.8881,Ya
1,Macro F1,0.8377,0.8377,Ya
2,Phish Recall,0.5316,0.5316,Ya


Baseline yang dilatih ulang menghasilkan nilai yang sama dengan catatan baseline, sehingga data training dan data testing pada notebook ini identik dengan baseline.

,Metric,Baseline (TF-IDF + MNB),MNB manual (count),Selisih (manual − baseline)
0,Accuracy,0.8881,0.9416,0.0535
1,Macro F1,0.8377,0.9337,0.0961
2,Ham Recall,0.9932,0.9832,-0.0100
3,Phish Recall,0.5316,0.9568,0.4252
4,Spam Recall,0.9051,0.8866,-0.0185
5,Ham hilang (Ham → Phish/Spam),174.0000,430.0000,256.0000
6,Phish → Ham,538.0000,24.0000,-514.0000
7,Spam → Ham,1991.0000,1301.0000,-690.0000
8,Phish → Spam,3475.0000,346.0000,-3129.0000


In [13]:
f1_gap = manual_summary['Macro F1'] - baseline_summary['Macro F1']
phish_gap = manual_summary['Phish Recall'] - baseline_summary['Phish Recall']
ham_lost_gap = manual_summary['Ham hilang (Ham → Phish/Spam)'] - baseline_summary['Ham hilang (Ham → Phish/Spam)']

if f1_gap > 0:
    direction = (
        f"Model manual lebih baik daripada baseline: Macro F1 naik {f1_gap:+.4f} "
        f"({baseline_summary['Macro F1']:.4f} → {manual_summary['Macro F1']:.4f}) "
        f"dan Phish Recall berubah {phish_gap:+.4f}. "
        'Kedua model memakai algoritma dan α yang sama; perbedaannya hanya pada representasi kata. '
        'Multinomial Naive Bayes memodelkan berapa kali kata muncul, sehingga frekuensi kata (*count*) adalah masukan alaminya. '
        'Dengan TF-IDF, total bobot setiap kelas jauh lebih kecil, sehingga tambahan smoothing α|V| meratakan '
        'probabilitas kata dan ciri khas kelas kecil seperti Phish melemah.'
    )
else:
    direction = (
        f"Model manual tidak lebih baik daripada baseline: Macro F1 berubah {f1_gap:+.4f} "
        f"({baseline_summary['Macro F1']:.4f} → {manual_summary['Macro F1']:.4f}) "
        f"dan Phish Recall berubah {phish_gap:+.4f}."
    )
ham_text = (
    f"Jumlah Ham yang hilang berubah dari {baseline_summary['Ham hilang (Ham → Phish/Spam)']:,} "
    f"menjadi {manual_summary['Ham hilang (Ham → Phish/Spam)']:,} email ({ham_lost_gap:+,})."
)
weakest = min(['Ham Recall', 'Phish Recall', 'Spam Recall'], key=lambda name: manual_summary[name])
weak_text = (
    f"Recall terendah model manual ada pada kelas {weakest.split()[0]} ({manual_summary[weakest]:.4f}); "
    f"kesalahan terbesarnya adalah Spam → Ham ({manual_summary['Spam → Ham']:,}) "
    f"dan Phish → Spam ({manual_summary['Phish → Spam']:,})."
)
display(Markdown(direction + ' ' + ham_text + ' ' + weak_text))

Model manual lebih baik daripada baseline: Macro F1 naik +0.0961 (0.8377 → 0.9337) dan Phish Recall berubah +0.4252. Kedua model memakai algoritma dan α yang sama; perbedaannya hanya pada representasi kata. Multinomial Naive Bayes memodelkan berapa kali kata muncul, sehingga frekuensi kata (*count*) adalah masukan alaminya. Dengan TF-IDF, total bobot setiap kelas jauh lebih kecil, sehingga tambahan smoothing α|V| meratakan probabilitas kata dan ciri khas kelas kecil seperti Phish melemah. Jumlah Ham yang hilang berubah dari 174 menjadi 430 email (+256). Recall terendah model manual ada pada kelas Spam (0.8866); kesalahan terbesarnya adalah Spam → Ham (1,301) dan Phish → Spam (346).

## D.4 Naive Bayes *Library* dan Manual secara Berdampingan

Tabel berikut menempatkan tiga model Multinomial Naive Bayes pada data testing yang sama:

1. **Library, TF-IDF**: `TfidfVectorizer` + `MultinomialNB` dari scikit-learn (baseline).
2. **Library, frekuensi kata**: `CountVectorizer` + `MultinomialNB` dari scikit-learn (model pembanding pada bagian C).
3. **Manual, frekuensi kata**: model tanpa library pada bagian B.

Kolom 2 dan 3 memakai algoritma, masukan, dan α yang sama, sehingga perbandingannya menguji implementasi. Kolom 1 dan 2 memakai algoritma yang sama dengan masukan berbeda, sehingga perbandingannya menguji representasi kata.

In [ ]:
library_summary = summarize(sk_pred)
side_by_side = pd.DataFrame({
    '1. Library, TF-IDF (baseline)': baseline_summary,
    '2. Library, frekuensi kata': library_summary,
    '3. Manual, frekuensi kata': manual_summary,
})
side_by_side.loc['Prediksi berbeda dari model manual'] = [
    int((baseline_pred != manual_pred).sum()),
    int((sk_pred != manual_pred).sum()),
    0,
]
count_rows = [
    'Ham hilang (Ham → Phish/Spam)', 'Phish → Ham', 'Spam → Ham', 'Phish → Spam',
    'Prediksi berbeda dari model manual'
]
display(side_by_side.apply(
    lambda row: row.map(lambda value: f'{int(value):,}' if row.name in count_rows else f'{value:.4f}'),
    axis=1,
).rename_axis('Metric').reset_index())

library_confusion = pd.DataFrame(
    confusion_matrix(y_true, sk_pred, labels=list(CLASS_IDS)),
    index=confusion_df.index, columns=confusion_df.columns,
)
display(pd.concat(
    {'2. Library, frekuensi kata': library_confusion, '3. Manual, frekuensi kata': confusion_df},
    axis=1,
))

timing = pd.DataFrame({
    'Implementasi': ['2. Library, frekuensi kata', '3. Manual, frekuensi kata'],
    'Pelatihan (detik)': [sk_train_seconds, manual_train_seconds],
    'Prediksi data testing (detik)': [sk_predict_seconds, manual_predict_seconds],
})
timing['Total (detik)'] = timing['Pelatihan (detik)'] + timing['Prediksi data testing (detik)']
display(timing.round(1))

In [ ]:
implementation_differences = int((sk_pred != manual_pred).sum())
representation_gap = library_summary['Macro F1'] - baseline_summary['Macro F1']
library_total = sk_train_seconds + sk_predict_seconds
manual_total = manual_train_seconds + manual_predict_seconds

if implementation_differences == 0:
    implementation_text = (
        f'Model library dan model manual dengan frekuensi kata menghasilkan prediksi yang sama pada seluruh '
        f'{len(manual_pred):,} email testing, sehingga confusion matrix dan seluruh metriknya sama. '
    )
else:
    implementation_text = (
        f'Model library dan model manual dengan frekuensi kata berbeda pada {implementation_differences:,} '
        f'email testing. '
    )
representation_text = (
    f"Selisih Macro F1 antara baseline dan model library dengan frekuensi kata adalah {representation_gap:+.4f} "
    f"({baseline_summary['Macro F1']:.4f} → {library_summary['Macro F1']:.4f}). "
    'Selisih ini berasal dari representasi kata, karena algoritma dan α kedua model sama. '
)
if abs(library_total - manual_total) <= 0.1 * max(library_total, manual_total):
    speed_verdict = 'waktunya hampir sama'
elif library_total < manual_total:
    speed_verdict = 'library lebih cepat'
else:
    speed_verdict = 'implementasi manual lebih cepat'
speed_text = (
    f'Pelatihan dan prediksi memakan {library_total:.1f} detik pada library dan {manual_total:.1f} detik '
    f'pada implementasi manual; {speed_verdict}.'
)
display(Markdown(implementation_text + representation_text + speed_text))

# E. Audit Perhitungan pada Satu Email

Tabel berikut merinci perhitungan untuk email pertama pada data testing: prior, $\log S_c$, posterior, dan faktor likelihood untuk sepuluh kata pertama yang dikenal model. Kolom terakhir menampilkan $P(w\mid c)$ dari scikit-learn untuk kata dan kelas yang sama. Seluruh kata yang dikenal model tetap dipakai dalam prediksi; sepuluh kata hanya ditampilkan sebagai contoh.

In [14]:
audit_record = results_df.iloc[0]
audit_pred, audit_scores, audit_probabilities, audit_known = predict_multinomial_nb(
    audit_record['text'], model, include_probabilities=True
)
audit_summary = pd.DataFrame([
    {
        'kelas': LABELS[class_id],
        'prior': model['class_document_count'][class_id] / model['D'],
        'log S_c (manual)': audit_scores[class_id],
        'log S_c (scikit-learn)': sk_log_scores[0, class_id],
        'posterior': audit_probabilities[class_id],
    }
    for class_id in CLASS_IDS
])
feature_index = count_vectorizer.vocabulary_
audit_factors = []
for token, frequency in list(audit_known.items())[:10]:
    for class_id in CLASS_IDS:
        token_class_count = model['token_count'][class_id].get(token, 0)
        audit_factors.append({
            'kelas': LABELS[class_id], 'token': token, 'f(w,x)': frequency,
            'F(w,c)': token_class_count, 'pembilang': token_class_count + ALPHA,
            'penyebut': model['likelihood_denominator'][class_id],
            'P(w|c) manual': (token_class_count + ALPHA) / model['likelihood_denominator'][class_id],
            'P(w|c) scikit-learn': math.exp(sk_model.feature_log_prob_[class_id, feature_index[token]]),
        })
print('Label aktual  :', LABELS[int(audit_record['label'])])
print('Prediksi      :', LABELS[audit_pred])
print('Token dikenal :', sum(audit_known.values()))
print('Cuplikan teks :', audit_record['text'][:220])
display(audit_summary)
display(pd.DataFrame(audit_factors))

Label aktual  : Spam
Prediksi      : Spam
Token dikenal : 43
Cuplikan teks : this one is being promoted take advantage symescapenumberol asvpcurr price escapenumber escapenumber escapenumber day target price escapenumberaction aggresive buy hold short term bullish insider buying alert asvp have r


,kelas,prior,log S_c (manual),log S_c (scikit-learn),posterior
0,Ham,0.457948,-383.602523,-383.602523,2.073744e-23
1,Phish,0.153572,-506.491159,-506.491159,8.849090e-77
2,Spam,0.388480,-331.372422,-331.372422,1.000000e+00


,kelas,token,"f(w,x)","F(w,c)",pembilang,penyebut,P(w|c) manual,P(w|c) scikit-learn
0,Ham,this,2,158752,158753.0,33850512.0,4.689826e-03,4.689826e-03
1,Phish,this,2,0,1.0,4713090.0,2.121750e-07,2.121750e-07
2,Spam,this,2,112783,112784.0,17725482.0,6.362817e-03,6.362817e-03
3,Ham,one,1,58686,58687.0,33850512.0,1.733711e-03,1.733711e-03
4,Phish,one,1,12503,12504.0,4713090.0,2.653037e-03,2.653037e-03
5,Spam,one,1,37509,37510.0,17725482.0,2.116162e-03,2.116162e-03
6,Ham,is,1,238606,238607.0,33850512.0,7.048845e-03,7.048845e-03
7,Phish,is,1,0,1.0,4713090.0,2.121750e-07,2.121750e-07
8,Spam,is,1,116251,116252.0,17725482.0,6.558468e-03,6.558468e-03
9,Ham,being,1,11198,11199.0,33850512.0,3.308369e-04,3.308369e-04


# F. Kesimpulan

Sel berikut merangkum jawaban atas kedua pertanyaan notebook ini.

In [15]:
verification_ok = (not failed_checks) and toy_ok and metrics_ok
lines_out = []
if verification_ok:
    lines_out.append(
        '1. **Perhitungan manual benar.** Hitungan tangan pada data kecil, kesembilan pemeriksaan model, '
        'dan seluruh metrik evaluasi sama dengan scikit-learn.'
    )
else:
    problems = []
    if not toy_ok:
        problems.append('hitungan tangan (B.2)')
    if failed_checks:
        problems.append('pemeriksaan model (C): ' + ', '.join(failed_checks))
    if not metrics_ok:
        problems.append('metrik evaluasi (D.2)')
    lines_out.append('1. **Perhitungan manual belum sepenuhnya benar.** Bagian yang berbeda: ' + '; '.join(problems) + '.')
lines_out.append(
    f"2. **Pada data testing yang sama dengan baseline**, model manual mencapai Accuracy {manual_summary['Accuracy']:.4f}, "
    f"Macro F1 {manual_summary['Macro F1']:.4f}, dan Phish Recall {manual_summary['Phish Recall']:.4f}, "
    f"dibandingkan baseline {baseline_summary['Accuracy']:.4f}, {baseline_summary['Macro F1']:.4f}, "
    f"dan {baseline_summary['Phish Recall']:.4f}."
)
lines_out.append(
    '3. Pembagian data ' + ('terbukti identik dengan baseline (D.3).' if split_identical
                            else 'tidak terbukti identik dengan baseline (D.3).')
)
display(Markdown('\n'.join(lines_out)))

1. **Perhitungan manual benar.** Hitungan tangan pada data kecil, kesembilan pemeriksaan model, dan seluruh metrik evaluasi sama dengan scikit-learn.
2. **Pada data testing yang sama dengan baseline**, model manual mencapai Accuracy 0.9416, Macro F1 0.9337, dan Phish Recall 0.9568, dibandingkan baseline 0.8881, 0.8377, dan 0.5316.
3. Pembagian data terbukti identik dengan baseline (D.3).

## Referensi Metode

- C. D. Manning, P. Raghavan, dan H. Schütze, *Introduction to Information Retrieval*, Bab 13: Text classification and Naive Bayes. Cambridge University Press, 2008.
- A. McCallum dan K. Nigam, "A comparison of event models for Naive Bayes text classification," *AAAI-98 Workshop on Learning for Text Categorization*, 1998.
- scikit-learn, [Multinomial Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html#multinomial-naive-bayes) dan [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).